# 00 - Arranque

Ejecutar las celdas en orden en un servidor de Colab (CPU, RAM alta). Ver `README.md`.

In [15]:
import os
import sys

REPO = "Adriel23456/Applied-AI-Agent-Project"
BRANCH = "03DataCollectionAndEnvironmentSetup"

if not os.path.exists("/content/repo"):
    !git clone -b {BRANCH} https://github.com/{REPO}.git /content/repo
else:
    !git -C /content/repo pull

%cd /content/repo
!pip install -q -r requirements.txt
sys.path.insert(0, "/content/repo/src")

Already up to date.
/content/repo


In [16]:
from ptcg_wp.colab import setup_kaggle

setup_kaggle()
!kaggle datasets list -s "pokemon tcg ai battle" | head -5

ref                                               title                                                                                size  lastUpdated                 downloadCount  voteCount  usabilityRating  
------------------------------------------------  -----------------------------------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
busyaprime/pokemon-tcg-ai-battle-live-meta        Pokemon TCG AI Battle: live ladder meta                                              2614  2026-08-01 16:00:59.037000            853         13  0.64705884       
kaggle/pokemon-tcg-ai-battle-episodes-index       The Pokémon Company - PTCG AI Battle Challenge Simulation Episodes Index             2606  2026-08-31 00:13:57.020000         104361         36  0.4117647        
citerne/pokemon-tcg-agent-showcase-media          Pokemon TCG Agent Showcase Media                                                 29061831  2026-08

## Ingesta

Descarga cada día de `configs/dias.yaml`, escribe un Parquet por día y borra el crudo.

In [17]:
!PYTHONPATH=src python -m ptcg_wp.ingest --config configs/dias.yaml --raw /content/raw --out /content/datos

Traceback (most recent call last):
  File "<frozen runpy>", line 203, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/repo/src/ptcg_wp/ingest.py", line 91, in <module>
    main()
    ~~~~^^
  File "/content/repo/src/ptcg_wp/ingest.py", line 87, in main
    ingest_day(day, args.raw, args.out, args.keep_raw)
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/repo/src/ptcg_wp/ingest.py", line 70, in ingest_day
    df = build_games_table(raw_day, day)
  File "/content/repo/src/ptcg_wp/ingest.py", line 60, in build_games_table
    rows = [game_row(p, day) for p in sorted(Path(raw_day_dir).glob("*.json"))]
            ~~~~~~~~^^^^^^^^
  File "/content/repo/src/ptcg_wp/ingest.py", line 35, in game_row
    g = json.load(f)
  File "/usr/lib/python3.13/json/__init__.py", line 298, in load
    return loads(fp.read(),
        cls=cls, object_hook=object_hook,
        parse_float=parse_float, parse_int=parse_int,
        parse_constant=par

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.concat(pd.read_parquet(p) for p in sorted(Path("/content/datos").glob("partidas_*.parquet")))
print(df.shape)
print(df["winner"].value_counts())
df.head()